# Tokenizer Fertility: Armenian vs English

Comparing how different tokenizers handle Armenian (low-resource, non-Latin script) vs English text.

Key finding: XLM-RoBERTa (SentencePiece) treats Armenian on par with English,
while OpenAI's cl100k_base (BPE) inflates Armenian by ~5x due to byte-level fallback.

In [2]:
from transformers import AutoTokenizer
import tiktoken
import pandas as pd

/Users/ashmelev/code/02_smartlabs/epg-embedding-benchmark/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Test phrases

Parallel EPG titles in Armenian, Russian, and English — including domain-specific abbreviation prefixes.

In [3]:
test_phrases = [
    {
        "id": "movie_secret_ararat",
        "en": "Feature Film: Secret of Ararat",
        "ru": "\u043a/\u0444 \u0422\u0430\u0439\u043d\u0430 \u0410\u0440\u0430\u0440\u0430\u0442\u0430",
        "hy": "\u0586/\u0586 \u0531\u0580\u0561\u0580\u0561\u057f\u056b \u0563\u0561\u0572\u057f\u0576\u056b\u0584\u0568",
    },
    {
        "id": "evening_news",
        "en": "Evening News",
        "ru": "\u0412\u0435\u0447\u0435\u0440\u043d\u0438\u0435 \u043d\u043e\u0432\u043e\u0441\u0442\u0438",
        "hy": "\u0535\u0580\u0565\u056f\u0578\u0575\u0561\u0576 \u056c\u0578\u0582\u0580\u0565\u0580",
    },
    {
        "id": "tv_series_prefix",
        "en": "TV Series: Dynasty",
        "ru": "\u0442/\u0441 \u0414\u0438\u043d\u0430\u0441\u0442\u0438\u044f",
        "hy": "\u0540/\u054d \u0534\u056b\u0576\u0561\u057d\u057f\u056b\u0561",
    },
    {
        "id": "live_football",
        "en": "Live Football",
        "ru": "\u041f\u0440\u044f\u043c\u0430\u044f \u0442\u0440\u0430\u043d\u0441\u043b\u044f\u0446\u0438\u044f \u0444\u0443\u0442\u0431\u043e\u043b\u0430",
        "hy": "\u0556\u0578\u0582\u057f\u0562\u0578\u056c\u056b \u0578\u0582\u0572\u056b\u0572 \u0570\u0565\u057c\u0561\u0580\u0571\u0561\u056f\u0578\u0582\u0574",
    },
    {
        "id": "animation_prefix",
        "en": "Animation: Tom and Jerry",
        "ru": "\u043c/\u0444 \u0422\u043e\u043c \u0438 \u0414\u0436\u0435\u0440\u0440\u0438",
        "hy": "\u0544/\u0556 \u0539\u0578\u0574 \u0587 \u054b\u0565\u0580\u056b",
    },
]

# Display
for p in test_phrases:
    print(f"{p['id']}:")
    print(f"  EN: {p['en']}")
    print(f"  RU: {p['ru']}")
    print(f"  HY: {p['hy']}")
    print()

movie_secret_ararat:
  EN: Feature Film: Secret of Ararat
  RU: к/ф Тайна Арарата
  HY: ֆ/ֆ Արարատի գաղտնիքը

evening_news:
  EN: Evening News
  RU: Вечерние новости
  HY: Երեկոյան լուրեր

tv_series_prefix:
  EN: TV Series: Dynasty
  RU: т/с Династия
  HY: Հ/Ս Դինաստիա

live_football:
  EN: Live Football
  RU: Прямая трансляция футбола
  HY: Ֆուտբոլի ուղիղ հեռարձակում

animation_prefix:
  EN: Animation: Tom and Jerry
  RU: м/ф Том и Джерри
  HY: Մ/Ֆ Թոմ և Ջերի



## 1. XLM-RoBERTa tokenizer (SentencePiece)

Used by: `intfloat/multilingual-e5-*`, `BAAI/bge-m3`, `sentence-transformers/LaBSE`

In [13]:
xlmr_tok = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
print(f"Tokenizer: {xlmr_tok.__class__.__name__}")
print(f"Vocab size: {xlmr_tok.vocab_size:,}")

Tokenizer: XLMRobertaTokenizerFast
Vocab size: 250,002


In [30]:
rows = []
for phrase in test_phrases:
    for lang in ["en", "ru", "hy"]:
        text = phrase[lang]
        ids = xlmr_tok.encode(text, add_special_tokens=False)
        tokens = xlmr_tok.convert_ids_to_tokens(ids)
        rows.append({
            "id": phrase["id"],
            "lang": lang.upper(),
            "text": text,
            "n_chars": len(text),
            "n_tokens": len(tokens),
            "tokens": tokens,
        })

df_xlmr = pd.DataFrame(rows)

# Show token counts
pivot = df_xlmr.pivot(index="id", columns="lang", values="n_tokens")
pivot["HY/EN"] = (pivot["HY"] / pivot["EN"]).round(2)
pivot["RU/EN"] = (pivot["RU"] / pivot["EN"]).round(2)
print("XLM-RoBERTa (SentencePiece) — token counts:")
print(pivot.to_string())
print(f"\nMean HY/EN ratio: {pivot['HY/EN'].mean():.2f}x")
print(f"Mean RU/EN ratio: {pivot['RU/EN'].mean():.2f}x")

XLM-RoBERTa (SentencePiece) — token counts:
lang                 EN  HY  RU  HY/EN  RU/EN
id                                           
animation_prefix      5   8   7   1.60    1.4
evening_news          3   4   3   1.33    1.0
live_football         2   9   7   4.50    3.5
movie_secret_ararat   7   6   7   0.86    1.0
tv_series_prefix      5   7   6   1.40    1.2

Mean HY/EN ratio: 1.94x
Mean RU/EN ratio: 1.62x


In [31]:
# Detailed token breakdown
for _, row in df_xlmr.iterrows():
    print(f"[{row['lang']}] {row['text']}")
    print(f"  -> {row['n_tokens']} tokens: {row['tokens']}")
    print()

[EN] Feature Film: Secret of Ararat
  -> 7 tokens: ['▁Feature', '▁Film', ':', '▁Secret', '▁of', '▁Ara', 'rat']

[RU] к/ф Тайна Арарата
  -> 7 tokens: ['▁к', '/', 'ф', '▁Тай', 'на', '▁Ара', 'рата']

[HY] ֆ/ֆ Արարատի գաղտնիքը
  -> 6 tokens: ['▁ֆ', '/', 'ֆ', '▁Արարատի', '▁գաղտնի', 'քը']

[EN] Evening News
  -> 3 tokens: ['▁Even', 'ing', '▁News']

[RU] Вечерние новости
  -> 3 tokens: ['▁Вечер', 'ние', '▁новости']

[HY] Երեկոյան լուրեր
  -> 4 tokens: ['▁Եր', 'եկ', 'ոյան', '▁լուրեր']

[EN] TV Series: Dynasty
  -> 5 tokens: ['▁TV', '▁Series', ':', '▁Dyna', 'sty']

[RU] т/с Династия
  -> 6 tokens: ['▁т', '/', 'с', '▁Дина', 'сти', 'я']

[HY] Հ/Ս Դինաստիա
  -> 7 tokens: ['▁Հ', '/', 'Ս', '▁Դ', 'ինա', 'ստի', 'ա']

[EN] Live Football
  -> 2 tokens: ['▁Live', '▁Football']

[RU] Прямая трансляция футбола
  -> 7 tokens: ['▁Прям', 'ая', '▁трансл', 'я', 'ция', '▁футбол', 'а']

[HY] Ֆուտբոլի ուղիղ հեռարձակում
  -> 9 tokens: ['▁Ֆ', 'ուտ', 'բոլ', 'ի', '▁ուղիղ', '▁', 'հեռ', 'արձակ', 'ում']

[EN] Animation: 

## 2. OpenAI tokenizer (cl100k_base / BPE)

Used by: `text-embedding-3-large`, `text-embedding-3-small`

In [32]:
oai_enc = tiktoken.encoding_for_model("text-embedding-3-large")
print(f"Encoding: {oai_enc.name}")

Encoding: cl100k_base


In [33]:
rows = []
for phrase in test_phrases:
    for lang in ["en", "ru", "hy"]:
        text = phrase[lang]
        ids = oai_enc.encode(text)
        # Decode individual tokens (may show replacement chars for byte tokens)
        token_strs = [oai_enc.decode([t]) for t in ids]
        rows.append({
            "id": phrase["id"],
            "lang": lang.upper(),
            "text": text,
            "n_chars": len(text),
            "n_tokens": len(ids),
            "tokens": token_strs,
        })

df_oai = pd.DataFrame(rows)

pivot = df_oai.pivot(index="id", columns="lang", values="n_tokens")
pivot["HY/EN"] = (pivot["HY"] / pivot["EN"]).round(2)
pivot["RU/EN"] = (pivot["RU"] / pivot["EN"]).round(2)
print("OpenAI cl100k_base (BPE) — token counts:")
print(pivot.to_string())
print(f"\nMean HY/EN ratio: {pivot['HY/EN'].mean():.2f}x")
print(f"Mean RU/EN ratio: {pivot['RU/EN'].mean():.2f}x")

OpenAI cl100k_base (BPE) — token counts:
lang                 EN  HY  RU  HY/EN  RU/EN
id                                           
animation_prefix      5  24  10   4.80   2.00
evening_news          3  29   7   9.67   2.33
live_football         2  50  14  25.00   7.00
movie_secret_ararat   7  37  10   5.29   1.43
tv_series_prefix      4  22   7   5.50   1.75

Mean HY/EN ratio: 10.05x
Mean RU/EN ratio: 2.90x


In [34]:
# Detailed token breakdown
for _, row in df_oai.iterrows():
    print(f"[{row['lang']}] {row['text']}")
    print(f"  -> {row['n_tokens']} tokens: {row['tokens']}")
    print()

[EN] Feature Film: Secret of Ararat
  -> 7 tokens: ['Feature', ' Film', ':', ' Secret', ' of', ' Ar', 'arat']

[RU] к/ф Тайна Арарата
  -> 10 tokens: ['к', '/', 'ф', ' Т', 'ай', 'на', ' А', 'р', 'ар', 'ата']

[HY] ֆ/ֆ Արարատի գաղտնիքը
  -> 37 tokens: ['�', '�', '/', '�', '�', ' ', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', ' ', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�']

[EN] Evening News
  -> 3 tokens: ['Even', 'ing', ' News']

[RU] Вечерние новости
  -> 7 tokens: ['В', 'еч', 'ер', 'ни', 'е', ' нов', 'ости']

[HY] Երեկոյան լուրեր
  -> 29 tokens: ['�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', ' ', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�', '�']

[EN] TV Series: Dynasty
  -> 4 tokens: ['TV', ' Series', ':', ' Dynasty']

[RU] т/с Династия
  -> 7 tokens: ['т', '/', 'с', ' Д', 'ина', 'ст', 'ия']

[HY] Հ/Ս Դինաստիա
  -> 22 tokens: ['�', '�', '/', '�', '�', ' ', '�', '�', '�', '�', '

## 3. Side-by-side comparison

In [35]:
comparison = []
for phrase in test_phrases:
    for lang in ["en", "ru", "hy"]:
        text = phrase[lang]
        n_xlmr = len(xlmr_tok.encode(text, add_special_tokens=False))
        n_oai = len(oai_enc.encode(text))
        comparison.append({
            "id": phrase["id"],
            "lang": lang.upper(),
            "chars": len(text),
            "XLM-R tokens": n_xlmr,
            "OpenAI tokens": n_oai,
            "OpenAI/XLM-R": round(n_oai / n_xlmr, 2) if n_xlmr > 0 else None,
        })

df_cmp = pd.DataFrame(comparison)
print(df_cmp.to_string(index=False))

print("\n--- Averages by language ---")
print(df_cmp.groupby("lang")[["XLM-R tokens", "OpenAI tokens", "OpenAI/XLM-R"]].mean().round(2).to_string())

                 id lang  chars  XLM-R tokens  OpenAI tokens  OpenAI/XLM-R
movie_secret_ararat   EN     30             7              7          1.00
movie_secret_ararat   RU     17             7             10          1.43
movie_secret_ararat   HY     20             6             37          6.17
       evening_news   EN     12             3              3          1.00
       evening_news   RU     16             3              7          2.33
       evening_news   HY     15             4             29          7.25
   tv_series_prefix   EN     18             5              4          0.80
   tv_series_prefix   RU     12             6              7          1.17
   tv_series_prefix   HY     12             7             22          3.14
      live_football   EN     13             2              2          1.00
      live_football   RU     25             7             14          2.00
      live_football   HY     26             9             50          5.56
   animation_prefix   EN 

## 4. Abbreviation-only tokenization

How many tokens does each tokenizer spend on just the abbreviation prefix?

In [36]:
abbreviations = {
    "\u0586/\u0586 (Feature Film, HY)": "\u0586/\u0586",
    "\u0540/\u054d (TV Series, HY)": "\u0540/\u054d",
    "\u0544/\u0556 (Animation, HY)": "\u0544/\u0556",
    "\u043a/\u0444 (Feature Film, RU)": "\u043a/\u0444",
    "\u0442/\u0441 (TV Series, RU)": "\u0442/\u0441",
    "\u043c/\u0444 (Animation, RU)": "\u043c/\u0444",
}

for label, abbr in abbreviations.items():
    xlmr_ids = xlmr_tok.encode(abbr, add_special_tokens=False)
    xlmr_tokens = xlmr_tok.convert_ids_to_tokens(xlmr_ids)
    oai_ids = oai_enc.encode(abbr)
    oai_tokens = [oai_enc.decode([t]) for t in oai_ids]
    print(f"{label}:  {abbr}")
    print(f"  XLM-R:  {len(xlmr_tokens)} tokens -> {xlmr_tokens}")
    print(f"  OpenAI: {len(oai_tokens)} tokens -> {oai_tokens}")
    print()

ֆ/ֆ (Feature Film, HY):  ֆ/ֆ
  XLM-R:  3 tokens -> ['▁ֆ', '/', 'ֆ']
  OpenAI: 5 tokens -> ['�', '�', '/', '�', '�']

Հ/Ս (TV Series, HY):  Հ/Ս
  XLM-R:  3 tokens -> ['▁Հ', '/', 'Ս']
  OpenAI: 5 tokens -> ['�', '�', '/', '�', '�']

Մ/Ֆ (Animation, HY):  Մ/Ֆ
  XLM-R:  3 tokens -> ['▁Մ', '/', 'Ֆ']
  OpenAI: 5 tokens -> ['�', '�', '/', '�', '�']

к/ф (Feature Film, RU):  к/ф
  XLM-R:  3 tokens -> ['▁к', '/', 'ф']
  OpenAI: 3 tokens -> ['к', '/', 'ф']

т/с (TV Series, RU):  т/с
  XLM-R:  3 tokens -> ['▁т', '/', 'с']
  OpenAI: 3 tokens -> ['т', '/', 'с']

м/ф (Animation, RU):  м/ф
  XLM-R:  3 tokens -> ['▁м', '/', 'ф']
  OpenAI: 3 tokens -> ['м', '/', 'ф']



## 5. Cost implications

OpenAI charges per token. If Armenian text is ~5x more tokens than English for the same semantic content, you pay ~5x more for worse quality.

In [37]:
# OpenAI text-embedding-3-large pricing: $0.13 per 1M tokens (as of 2025)
PRICE_PER_M_TOKENS = 0.13

total_by_lang = df_oai.groupby("lang")["n_tokens"].sum()
total_en = total_by_lang["EN"]

print("Cost multiplier vs English (OpenAI text-embedding-3-large):")
for lang in ["EN", "RU", "HY"]:
    ratio = total_by_lang[lang] / total_en
    print(f"  {lang}: {ratio:.2f}x ({total_by_lang[lang]} tokens vs {total_en} EN tokens)")

print(f"\nFor 100K EPG titles:")
for lang in ["EN", "RU", "HY"]:
    avg_tokens = df_oai[df_oai["lang"] == lang]["n_tokens"].mean()
    cost = (avg_tokens * 100_000 / 1_000_000) * PRICE_PER_M_TOKENS
    print(f"  {lang}: ~{avg_tokens:.0f} tokens/title, ${cost:.2f} per 100K titles")

Cost multiplier vs English (OpenAI text-embedding-3-large):
  EN: 1.00x (21 tokens vs 21 EN tokens)
  RU: 2.29x (48 tokens vs 21 EN tokens)
  HY: 7.71x (162 tokens vs 21 EN tokens)

For 100K EPG titles:
  EN: ~4 tokens/title, $0.05 per 100K titles
  RU: ~10 tokens/title, $0.12 per 100K titles
  HY: ~32 tokens/title, $0.42 per 100K titles
